#### 7. Extend the gold layer with a second aggregation for a different stakeholder (e.g., the inventory team) and justify why it belongs in gold rather than being computed ad hoc by that team.

In [0]:
%sql
-- First check the data
select * from samples.tpch.orders limit 10

In [0]:
%sql
-- Pre-aggregated metrics for inventory planning 
-- 
-- Why this belongs in GOLD rather than ad-hoc queries:
-- 1. STANDARDIZED METRICS: Ensures consistent definitions of "urgent orders", "delayed shipments"
--    across all inventory dashboards and reports (single source of truth)
-- 2. PERFORMANCE: Aggregating 1.5M+ orders daily is expensive; pre-computing saves
--    repeated scans when inventory managers refresh dashboards throughout the day
-- 3. BUSINESS LOGIC: Embeds domain rules (e.g., critical order thresholds, priority
--    definitions) that shouldn't be reimplemented by each analyst
-- 4. GOVERNANCE: Provides auditable, version-controlled metrics for compliance
-- 5. CROSS-TEAM ALIGNMENT: Sales, logistics, and inventory all reference the same
--    aggregated numbers, preventing "whose numbers are right?" debates

CREATE OR REPLACE TABLE cyntexa_dev.gold.inventory_order_metrics AS
SELECT 
  o.o_orderpriority,
  o.o_orderstatus,
  DATE_TRUNC('month', o.o_orderdate) AS order_month,
  
  -- Volume metrics
  COUNT(*) AS total_orders,
  COUNT(DISTINCT o.o_custkey) AS unique_customers,
  
  -- Financial metrics
  SUM(o.o_totalprice) AS total_order_value,
  AVG(o.o_totalprice) AS avg_order_value,
  
  
  -- Priority segmentation for capacity planning
  SUM(CASE WHEN o.o_orderpriority IN ('1-URGENT', '2-HIGH') THEN o.o_totalprice ELSE 0 END) AS high_priority_value,
  
  -- Data quality
  current_timestamp() AS last_refreshed
  
FROM samples.tpch.orders o
WHERE o.o_orderdate >= '1992-01-01'  -- Scoping to available data range
GROUP BY 
  o.o_orderpriority,
  o.o_orderstatus,
  DATE_TRUNC('month', o.o_orderdate)
ORDER BY 
  order_month DESC,
  o.o_orderpriority;

In [0]:
%sql
select * from cyntexa_dev.gold.inventory_order_metrics 

#### 8. Write a short design note on which parts of this pipeline should run in the customer's data plane vs. rely on Databricks' control plane, and what that means for a network/security review.


## Data Plane vs. Control Plane Design Note

### Architecture Overview

For this medallion pipeline (Bronze → Silver → Gold), the workload distribution should be:

#### **Customer's Data Plane **
- **All data processing and storage**
  - Bronze ingestion from `samples.tpch.orders`
  - Silver transformations and aggregations
  - Gold aggregated tables (`inventory_order_metrics`)
  - Unity Catalog metastore 
  - All compute clusters running SQL/Spark queries
  
- **Why**: Customer data never leaves their network boundary; maintains data residency and compliance requirements

#### **Control Plane (Databricks-Managed)**
- **Orchestration and management**
  - Job scheduling and workflow orchestration
  - Notebook versioning and collaborative editing
  - Cluster lifecycle management (start/stop/scale)
  - Query history and audit logs
  - Unity Catalog metadata (table schemas, lineage, policies)
  - Access control and identity federation

- **Why**: Reduces operational overhead; Databricks handles Access Control, disaster recovery, and platform updates

---

### Network & Security Review Considerations

#### **1. Outbound Connectivity (Data Plane → Control Plane)**
- **Required**: Clusters need HTTPS (443) access to Databricks control plane for:
  - Heartbeats and telemetry
  - Retrieving job definitions and notebook code
  - Pushing logs and metrics
  
- **Security posture**: Use PrivateLink (AWS) / Private Endpoint (Azure) to avoid public internet traversal

#### **2. Data Exfiltration Prevention**
- **No customer data flows to control plane**: Only metadata (table names, schemas, lineage) is sent
- **Validation**: Network logs should show no large payloads (>1MB) to Databricks control plane IPs
- **Policy enforcement**: Unity Catalog policies run in data plane compute; blocked queries never send data out

#### **3. Source System Access**
- This pipeline reads from `samples.tpch` (Databricks-provided sample data in data plane)
- **Production equivalent**: If replacing with customer DB (on-prem SQL Server, Salesforce, etc.):
  - Clusters need network path to source (VPN, Direct Connect, or firewall rules)
  - Credentials stored in Databricks Secrets (encrypted at rest in control plane, decrypted in data plane at runtime)

#### **4. Compliance Boundary**
- **Data residency**: All PII/sensitive data stays in customer's region/VPC
- **Audit trail**: Job run logs in control plane (safe—no sensitive data), query results in data plane
- **Break-glass scenario**: Databricks support cannot access data plane without explicit customer approval (IP allowlisting + temp credentials)


#### 9. (Data Analyst) Build a query or lightweight dashboard directly against the gold table, and identify one data-quality issue you can trace back to a specific bronze or silver transformation decision.

In [0]:
%sql
-- Count the number of distinct products in gold layer to check the data quality
select count(distinct product_id) from cyntexa_dev.gold.total_revenue_by_products 

In [0]:
%sql
-- Now, we'll check at silver layer to see if the data quality is maintained
select count(distinct product_id) from cyntexa_dev.silver.sales_clean

In [0]:
%sql
-- Finally, from the bronze layer to see if the data quality is maintained
select count(distinct product_id) from cyntexa_dev.bronze.sales_raw